In [12]:
import os
import numpy as np

BASE_DIR = "recognised/team_members"
output_file = "recognised/team_members/encodings.npz"

all_encodings = []

# Loop through all .npy files in BASE_DIR
for file in os.listdir(BASE_DIR):
    if file.endswith(".npy"):
        path = os.path.join(BASE_DIR, file)
        try:
            encoding = np.load(path)
            all_encodings.append(encoding)
            print(f"Loaded: {file}")
        except Exception as e:
            print(f"Error loading {file}: {e}")

if all_encodings:
    np.savez_compressed(output_file, encodings=np.array(all_encodings))
    print(f"\nSaved {len(all_encodings)} encodings into {output_file}")
else:
    print("No .npy files found in", BASE_DIR)


Loaded: 20251031_015522_encoding.npy
Loaded: 20251031_015429_encoding.npy
Loaded: 20251031_013048_encoding.npy
Loaded: 20251031_015452_encoding.npy
Loaded: 20251031_013202_encoding.npy
Loaded: 20251031_015407_encoding.npy
Loaded: 20251031_014538_encoding.npy
Loaded: 20251031_014046_encoding.npy
Loaded: 20251031_014703_encoding.npy
Loaded: 20251031_015252_encoding.npy
Loaded: 20251031_014517_encoding.npy
Loaded: 20251031_013152_encoding.npy
Loaded: 20251031_013058_encoding.npy
Loaded: 20251031_013130_encoding.npy
Loaded: 20251031_014653_encoding.npy
Loaded: 20251031_014025_encoding.npy
Loaded: 20251031_014714_encoding.npy
Loaded: 20251031_014559_encoding.npy
Loaded: 20251031_014643_encoding.npy
Loaded: 20251031_015303_encoding.npy
Loaded: 20251031_013234_encoding.npy
Loaded: 20251031_014621_encoding.npy
Loaded: 20251031_013244_encoding.npy
Loaded: 20251031_015543_encoding.npy
Loaded: 20251031_013109_encoding.npy
Loaded: 20251031_014528_encoding.npy
Loaded: 20251031_013141_encoding.npy
L

In [13]:
import os
import cv2
import numpy as np
from insightface.app import FaceAnalysis

# --- CONFIG ---
ENCODINGS_FILE = "recognised/team_members/encodings.npz"
TEST_IMAGE_PATH = "sample3.jpeg"   # <-- change this to your sample image path
SIMILARITY_THRESHOLD = 0.35
# ---------------

# Load known encodings
if not os.path.exists(ENCODINGS_FILE):
    raise FileNotFoundError(f"Encodings file not found: {ENCODINGS_FILE}")

data = np.load(ENCODINGS_FILE, allow_pickle=True)
known_encodings = data["encodings"]
photo_names = data.get("names", [f"user_{i}" for i in range(len(known_encodings))])
print(f"Loaded {len(known_encodings)} known encodings from {ENCODINGS_FILE}")

# Initialize InsightFace model
print("Loading ArcFace model...")
face_app = FaceAnalysis(name="buffalo_l", root="/home/neub1t/.insightface")
face_app.prepare(ctx_id=0, det_size=(640, 640))
# face_app = FaceAnalysis(name="buffalo_l", root="/home/neub1t/.insightface")
# face_app.prepare(ctx_id=0, det_size=(320, 320), models=['detection', 'recognition'])

print("Model ready!")

# Read the test image
if not os.path.exists(TEST_IMAGE_PATH):
    raise FileNotFoundError(f"Test image not found: {TEST_IMAGE_PATH}")

img = cv2.imread(TEST_IMAGE_PATH)
faces = face_app.get(img)

if len(faces) == 0:
    print("No face detected in test image.")
    exit()

test_encoding = faces[0].embedding
test_norm = np.linalg.norm(test_encoding)

# Compute cosine similarity with all stored encodings
sims = np.dot(known_encodings, test_encoding) / (
    np.linalg.norm(known_encodings, axis=1) * test_norm
)

# Find the best match
best_index = int(np.argmax(sims))
best_score = float(sims[best_index])
matched_name = str(photo_names[best_index])

print("\n --- Comparison Result ---")
print(f"Best Match: {matched_name}")
print(f"Similarity Score: {best_score:.4f}")

if best_score > SIMILARITY_THRESHOLD:
    print("Face MATCHED (authorised)")
else:
    print("No Match (unauthorised)")


Loaded 39 known encodings from recognised/team_members/encodings.npz
Loading ArcFace model...
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/neub1t/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/neub1t/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/neub1t/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)
Model ready!

 --- Comparison Result ---
Best Match: user_23
Similarity Score: 0.7784
Face MATCHED (authorised)


In [2]:
import os
import cv2
import numpy as np
from insightface.app import FaceAnalysis
import onnxruntime as ort


# --- ONNX Runtime threading setup ---
# print(" Configuring ONNX Runtime for 6 CPU cores...")
# sess_options = ort.SessionOptions()
# sess_options.intra_op_num_threads = 4   # parallel threads per operation
# sess_options.inter_op_num_threads = 2   # parallel operations allowed
# sess_options.execution_mode = ort.ExecutionMode.ORT_PARALLEL
# os.environ["OMP_NUM_THREADS"] = "6"     # ensures OpenMP respects 6 threads
# print(" ONNX Runtime configured for 6 threads.\n")


# --- CONFIG ---
ENCODINGS_FILE = "recognised/team_members/encodings.npz"
TEST_IMAGES = ["sample1.jpeg", "sample2.jpeg", "sample3.jpeg", "ptest1.jpeg", "ptest2.jpeg"]   # list of test image paths
SIMILARITY_THRESHOLD = 0.35
# ---------------

# --- Load known encodings ---
if not os.path.exists(ENCODINGS_FILE):
    raise FileNotFoundError(f"Encodings file not found: {ENCODINGS_FILE}")

data = np.load(ENCODINGS_FILE, allow_pickle=True)
known_encodings = data["encodings"]
photo_names = data.get("names", [f"user_{i}" for i in range(len(known_encodings))])
print(f"Loaded {len(known_encodings)} known encodings from {ENCODINGS_FILE}")

# --- Initialize ArcFace model once ---
print("Loading ArcFace model...")
face_app = FaceAnalysis(name="buffalo_l", root="/home/neub1t/.insightface")
face_app.prepare(ctx_id=0, det_size=(480, 480))
print("Model ready!\n")

# --- Process each image one by one ---
for test_image_path in TEST_IMAGES:
    print(f"\nChecking image: {test_image_path}")

    if not os.path.exists(test_image_path):
        print(f"Test image not found: {test_image_path}")
        continue

    img = cv2.imread(test_image_path)
    faces = face_app.get(img)

    if len(faces) == 0:
        print("No face detected in this image.")
        continue

    test_encoding = faces[0].embedding
    test_norm = np.linalg.norm(test_encoding)

    # Compute cosine similarity
    known_norms = np.linalg.norm(known_encodings, axis=1)
    sims = np.dot(known_encodings, test_encoding) / (known_norms * test_norm)

    best_index = int(np.argmax(sims))
    best_score = float(sims[best_index])
    matched_name = str(photo_names[best_index])

    print("--- Comparison Result ---")
    print(f"Best Match: {matched_name}")
    print(f"Similarity Score: {best_score:.4f}")

    if best_score > SIMILARITY_THRESHOLD:
        print("Face MATCHED (authorised)")
    else:
        print("No Match (unauthorised)")


Loaded 39 known encodings from recognised/team_members/encodings.npz
Loading ArcFace model...
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/neub1t/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/neub1t/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/neub1t/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (480, 480)
Model ready!


Checking image: sample1.jpeg
--- Comparison Result ---
Best Match: user_2
Similarity Score: 0.2808
No Match (unauthorised)

Checking image: sample2.jpeg
--- Comparison Result ---
Best Match: user_35
Similarity Score: 0.3150
No Match (unauthorised)

Checking image:

In [ ]:
import numpy as np
np.__config__.show()


Build Dependencies:
  blas:
    detection method: pkgconfig
    found: true
    include directory: /home/neub1t/anaconda3/envs/ml/include
    lib directory: /home/neub1t/anaconda3/envs/ml/lib
    name: openblas
    openblas configuration: USE_64BITINT=0 DYNAMIC_ARCH=1 DYNAMIC_OLDER= NO_CBLAS=
      NO_LAPACK=0 NO_LAPACKE= NO_AFFINITY=1 USE_OPENMP=0 PRESCOTT MAX_THREADS=128
    pc file directory: /home/neub1t/anaconda3/envs/ml/lib/pkgconfig
    version: 0.3.29
  lapack:
    detection method: internal
    found: true
    include directory: unknown
    lib directory: unknown
    name: dep140610056370512
    openblas configuration: unknown
    pc file directory: unknown
    version: 1.26.4
Compilers:
  c:
    args: -march=nocona, -mtune=haswell, -ftree-vectorize, -fPIC, -fstack-protector-strong,
      -fno-plt, -O2, -ffunction-sections, -pipe, -isystem, /home/neub1t/anaconda3/envs/ml/include,
      -fdebug-prefix-map=/croot/numpy_and_numpy_base_1755590845055/work=/usr/local/src/conda/numpy